# Bước 4 trên Kaggle — Phạm Văn Hoàng Anh Tú (2A202602507)

Chạy **test đúng một lần mỗi seed** cho chung kết F01 (phương pháp suy luận đã chốt ở Bước 3 trên val:
`I04_res288` + temperature scaling khớp trên val) và mốc T00 + I00, rồi chấm bằng `eval.py score` và `eval.py grade`.

**Trước khi chạy:** bấm **Add Input** → tab **Your Work** → gắn CẢ HAI notebook:
- notebook Bước 2 (có `deepweeds_runs/T00/seed0..2/best.pt`)
- notebook Bước 2b (có `deepweeds_runs/F01/seed0..2/best.pt`)

Kết quả nhỏ được gói vào `/kaggle/working/kaggle_buoc4.zip`.

In [ ]:
# Ô K1 — kiểm tra GPU, cài thư viện, in phiên bản
import os, sys, platform, subprocess
LOCAL_TEST = os.environ.get("LAB_LOCAL_TEST") == "1"   # chỉ Claude dùng khi chạy thử trên CPU; trên Kaggle luôn False
if not LOCAL_TEST:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "timm", "openpyxl"], check=True)
import numpy as np, pandas as pd, torch, torchvision, timm
print("python", platform.python_version(), "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| timm", timm.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
assert LOCAL_TEST or torch.cuda.is_available(), "Chưa bật GPU: Settings → Accelerator → GPU T4 x2"

In [ ]:
# Ô K2 — lấy code từ GitHub (nhánh có code đã chạy thử)
ROOT = os.environ.get("LAB_ROOT", "/kaggle")
WORK = f"{ROOT}/working"
REPO = f"{WORK}/repo"
BRANCH = "claude/project-thread-mdpq02"
if not LOCAL_TEST and not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    "https://github.com/tubepvhat1604-bot/K4-DAY02-PhamVanHoangAnhTu-2A202602507.git", REPO], check=True)
CODE_DIR = f"{REPO}/submissions/2A202602507_pham_van_hoang_anh_tu/code"
for p in (REPO, CODE_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)
OUT = f"{WORK}/sub"                      # giống cấu trúc thư mục bài nộp: curves/, predictions/, logs/
RUNS_DIR = f"{WORK}/deepweeds_runs"      # checkpoint + log từng run
for d in ("curves", "predictions", "logs"):
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
os.chdir(OUT)
import train, dataset
from pathlib import Path
from train import Config, run
print("code:", CODE_DIR)

In [ ]:
# Ô K3 — tải ảnh (Zenodo) + CSV fold 0 (GitHub tác giả), kiểm tra MD5, giải nén ra đĩa tạm
import hashlib, zipfile, urllib.request
DATA_DIR   = os.environ.get("LAB_DATA", "/tmp/data")
LABELS_DIR = f"{DATA_DIR}/labels"
ZIP_LOCAL  = f"{DATA_DIR}/images.zip"
ZIP_URL    = "https://zenodo.org/records/7939060/files/images.zip?download=1"
ZIP_MD5    = "b7b30f96d466fba86016aa5a26606e0f"
os.makedirs(LABELS_DIR, exist_ok=True)

def md5sum(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not LOCAL_TEST and not os.path.exists(f"{DATA_DIR}/.images_ok"):
    if not os.path.exists(ZIP_LOCAL):
        print("Tải images.zip từ Zenodo...")
        urllib.request.urlretrieve(ZIP_URL, ZIP_LOCAL)
    got = md5sum(ZIP_LOCAL)
    assert got == ZIP_MD5, f"MD5 sai: {got}"
    print("MD5 đúng:", got)
    with zipfile.ZipFile(ZIP_LOCAL) as z:
        z.extractall(f"{DATA_DIR}/images")
    open(f"{DATA_DIR}/.images_ok", "w").close()

BASE_URL = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    if not os.path.exists(f"{LABELS_DIR}/{name}.csv"):
        urllib.request.urlretrieve(f"{BASE_URL}/{name}.csv", f"{LABELS_DIR}/{name}.csv")

IMAGES_DIR = max((r for r, _, fs in os.walk(f"{DATA_DIR}/images")), key=lambda r: len(os.listdir(r)))
print("IMAGES_DIR =", IMAGES_DIR, "|", sum(f.endswith(".jpg") for f in os.listdir(IMAGES_DIR)), "ảnh .jpg")
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
dataset.check_split(train_df, val_df, test_df, IMAGES_DIR,
                    expected_total=None if LOCAL_TEST else dataset.TOTAL_IMAGES, verbose=False)
print("Kiểm tra chia dữ liệu: ĐẠT")

In [ ]:
# Ô K4 — tìm checkpoint T00 và F01 trong Input, gom về một thư mục (liên kết, không chép)
import glob, json
INPUT_ROOT = os.environ.get("LAB_INPUT", f"{ROOT}/input")
IN_RUNS = f"{WORK}/runs_in"
os.makedirs(IN_RUNS, exist_ok=True)
for exp in ("T00", "F01"):
    hits = sorted(glob.glob(f"{INPUT_ROOT}/**/deepweeds_runs/{exp}/seed0/best.pt", recursive=True))
    assert hits, f"Chưa thấy {exp}/seed0/best.pt: bấm Add Input → Your Work → gắn notebook có {exp}, rồi chạy lại"
    src = Path(hits[0]).parents[1]
    for k in (0, 1, 2):
        assert (src / f"seed{k}" / "best.pt").exists(), f"Thiếu {exp}/seed{k}/best.pt trong {src}"
    if not os.path.exists(f"{IN_RUNS}/{exp}"):
        os.symlink(src, f"{IN_RUNS}/{exp}")
    print(exp, "lấy từ:", src)

# Phương pháp chốt ở Bước 3 (chép nguyên logs/buoc3_choice.json của lần chạy Bước 3 trên Kaggle)
CHOICE = {"method": "I04_res288", "loader": "288", "views": ["id"], "aggregate": "logit",
          "val_macro_f1_mean_3seed": 0.9784763969487772, "plus_temperature_scaling": True,
          "realtime_p95_b1_ms": 7.8901769999959015,
          "rule": "max macro-F1 val trung bình 3 seed trong các phương pháp một mô hình; hòa thì ít view hơn"}
Path(OUT, "logs", "buoc3_choice.json").write_text(json.dumps(CHOICE, indent=2, ensure_ascii=False))
print("Phương pháp chung kết:", CHOICE["method"], "+ temperature scaling")

In [ ]:
# Ô K5 — TEST: chạy MỘT lần mỗi seed (đã có file *_test.csv thì dừng, không chạy lại)
import buoc4
info = buoc4.run_test(IN_RUNS, IMAGES_DIR, LABELS_DIR, OUT, num_workers=0 if LOCAL_TEST else 2)
print("Nhiệt độ T (khớp trên val):", {k: round(v, 3) for k, v in info["temperature"].items()})
print(sorted(os.listdir(f"{OUT}/predictions")))

In [ ]:
# Ô K6 — chấm bằng eval.py: score cho F01, F01 chưa TS, T00; grade phần I (I1-I5)
EVAL = f"{REPO}/eval.py"
TEST_CSV, VAL_CSV, LABELS_CSV = f"{LABELS_DIR}/test_subset0.csv", f"{LABELS_DIR}/val_subset0.csv", f"{LABELS_DIR}/labels.csv"
os.makedirs(f"{OUT}/logs/eval", exist_ok=True)

def sh(args, log):
    r = subprocess.run([sys.executable, EVAL] + args, capture_output=True, text=True)
    Path(OUT, "logs", "eval", log).write_text(r.stdout + r.stderr)
    print(r.stdout + r.stderr)
    assert r.returncode == 0, f"eval.py lỗi (mã {r.returncode})"

P = f"{OUT}/predictions"
for tag, pat in (("F01", "F01_seed*_test.csv"), ("F01_uncal", "F01_uncal_seed*_test.csv"), ("T00", "T00_seed*_test.csv")):
    sh(["score", "--pred", f"{P}/{pat}", "--test-csv", TEST_CSV, "--labels", LABELS_CSV,
        "--tag", tag, "--out", f"{OUT}/logs/eval"], f"score_{tag}.txt")
sh(["grade", "--final", f"{P}/F01_seed*_test.csv", "--baseline", f"{P}/T00_seed*_test.csv",
    "--uncal", f"{P}/F01_uncal_seed*_test.csv", "--final-val", f"{P}/F01_seed*_val.csv",
    "--latency-p95-ms", f"{CHOICE['realtime_p95_b1_ms']:.2f}", "--latency-method", "proper",
    "--test-csv", TEST_CSV, "--val-csv", VAL_CSV, "--labels", LABELS_CSV, "--out", f"{OUT}/logs/eval"], "grade.txt")

In [ ]:
# Ô K7 — ma trận nhầm lẫn test (cộng 3 seed) cho F01 và T00
for exp in ("F01", "T00"):
    buoc4.plot_confusion(f"{OUT}/predictions/{exp}_seed*_test.csv", LABELS_DIR,
                         f"{OUT}/curves/confusion_test_{exp}.png", f"Test {exp}")
print(sorted(os.listdir(f"{OUT}/curves")))

In [ ]:
# Ô K8 — gói kết quả Bước 4 thành kaggle_buoc4.zip (dự đoán, log chấm điểm, hình)
import zipfile
zp = Path(WORK) / "kaggle_buoc4.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for sub in ("predictions", "logs", "curves"):
        for f in sorted(Path(OUT, sub).rglob("*")):
            if f.is_file():
                z.write(f, Path("sub") / f.relative_to(OUT))
print(zp, f"{zp.stat().st_size / 1e3:.0f} KB")